# GigaAM: микрофон → текст (RU / KK / EN)

Запустите ячейки **1–3 по очереди**, затем в ячейке 3 нажмите **Start**, разрешите микрофон, произнесите фразу и нажмите **Stop**. Готовый аудиофайл не нужен. Повторный запуск ячейки 3 начинает новую запись.

Это **запись, затем распознавание**, а не потоковые частичные результаты. Используется тот же локальный backend GigaAM-Multilingual (~220M), что и в сервисе; внешних STT API нет. Один многоязычный декодер, без принудительного выбора языка и без автоматических языковых меток. Качество RU, KK, EN и смешанной речи нужно оценить на ваших примерах; этот notebook не является бенчмарком.

**Приватность:** в Colab звук передаётся из браузера в виртуальную машину Google. Это не полностью локальное выполнение на вашем компьютере. Используйте только неконфиденциальные тестовые фразы и записывайте других людей только с их согласия. Google Drive не подключается. Временные аудиофайлы удаляются после распознавания, текст остаётся в памяти и выводе notebook до очистки. Не сохраняйте/не публикуйте notebook с личными выводами; ниже есть очистка. Отключение runtime не заменяет политику хранения Google.

В Colab выберите **Runtime / Среда выполнения → Change runtime type → T4 GPU**, если GPU доступен. GPU не гарантирован; предусмотрен CPU. Установка и загрузка весов (~1 ГБ плюс зависимости) занимают несколько минут. Начните с 5–10 секунд речи. Нужен браузер с `MediaRecorder` (например, актуальный Chrome), HTTPS и разрешение микрофона.

[Исходный сервис](https://github.com/AssylbekGizatov/teams-local-transcriber) · [Модель и лицензия](https://huggingface.co/ai-sage/GigaAM-Multilingual)

## 1. Установка в отдельное Python-окружение
Код сервиса и GigaAM закреплены на конкретных коммитах, веса проверяются SHA256. Устанавливаются пакеты из PyPI, официальные PyTorch wheels и закреплённый исходный код GigaAM из GitHub. Ячейка не принимает сторонние checkpoint-файлы и не использует `trust_remote_code`.

Если GPU не нужен или установка CUDA не удаётся, выставьте `USE_GPU = False` и повторите ячейку. После смены типа runtime выполните 1–3 снова. Требуется Python 3.11–3.13; проверенная версия проекта — 3.12.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

USE_GPU = True  # False: только CPU, меньше скачиваний
REPO_URL = "https://github.com/AssylbekGizatov/teams-local-transcriber.git"
REPO_REVISION = "dd7b94a6c680d1ca891eeaa799bd68d70cf2559e"
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Нужен Colab runtime с Python 3.11–3.13 (проверено на 3.12).")
try:
    from google.colab import output
except ImportError as exc:
    raise RuntimeError("Откройте этот notebook в Google Colab.") from exc

BASE = Path("/content/gigaam-microphone-demo")
REPO = BASE / "service"
VENV = BASE / "venv"
BASE.mkdir(parents=True, exist_ok=True)

def run(command, **kwargs):
    subprocess.run([str(x) for x in command], check=True, **kwargs)

if not shutil.which("ffmpeg"):
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "ffmpeg"])
if not (REPO / ".git").is_dir():
    run(["git", "clone", "--no-checkout", REPO_URL, REPO])
run(["git", "-C", REPO, "fetch", "origin", REPO_REVISION])
run(["git", "-C", REPO, "checkout", "--detach", REPO_REVISION])
actual_revision = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
assert actual_revision == REPO_REVISION, "Не совпадает закреплённая версия сервиса"
if not (VENV / "bin/python").is_file():
    run([sys.executable, "-m", "venv", "--without-pip", VENV])
PYTHON = VENV / "bin/python"
# pip управляет отдельным venv даже если в Colab нет ensurepip.
PIP = [sys.executable, "-m", "pip", "--python", str(PYTHON)]
# Не импортируем torch в ядро Colab: весь ASR работает в отдельном venv.
has_gpu = bool(shutil.which("nvidia-smi")) and subprocess.run(
    ["nvidia-smi", "-L"], capture_output=True
).returncode == 0
wheel_channel = "cu128" if USE_GPU and has_gpu else "cpu"
run([*PIP, "install", f"torch==2.10.0+{wheel_channel}", f"torchaudio==2.10.0+{wheel_channel}",
     "--index-url", f"https://download.pytorch.org/whl/{wheel_channel}"])
run([*PIP, "install", "-r", REPO / "requirements-asr.lock"])
run([*PIP, "check"])
print("Установка завершена. PyTorch wheels:", wheel_channel)

## 2. Загрузка и проверка модели
После первого запуска модель уже есть в текущем runtime; повторная проверка не скачивает неизменённые веса. Для распознавания сеть не нужна. Если GPU недоступен для PyTorch, используется CPU; при нехватке памяти GPU установите `DEVICE = "cpu"` и повторите эту ячейку.

In [ ]:
import json

MODEL = BASE / "models/gigaam"
run([PYTHON, REPO / "scripts/download_models.py", "--backend", "gigaam", "--output", MODEL])
DEVICE = "auto" if USE_GPU else "cpu"  # можно явно поставить "cpu"
probe = """
import json, sys
from pathlib import Path
from app.asr import ASRSettings, LocalASR
asr = LocalASR(ASRSettings(model_path=Path(sys.argv[1]), device=sys.argv[2], cpu_threads=2))
backend = asr._get_backend()
print(json.dumps({"device": str(backend.model._device)}))
"""
check = subprocess.run([str(PYTHON), "-c", probe, str(MODEL), DEVICE],
                       cwd=REPO, text=True, capture_output=True, timeout=300)
if check.returncode:
    raise RuntimeError("Модель не загрузилась. Проверьте установку; попробуйте DEVICE = 'cpu'. "
                       "Если runtime завершён из-за памяти, переподключитесь и повторите 1–2.")
print("Модель проверена и загружается. Устройство:", json.loads(check.stdout)["device"])
print("Можно запускать запись. Модель загружается в отдельный процесс для каждой записи.")

## 3. Записать и распознать
Нажмите ▶ у ячейки, затем **Start** в появившемся блоке. Нажмите **Stop** после фразы. Автостоп через 30 секунд; максимум 5 МиБ. Пока ожидается разрешение браузера, Stop отменяет операцию. При прерывании ячейки закройте вывод/вкладку или запретите микрофон в браузере, если индикатор ещё горит; автостоп остаётся страховкой.

Если разрешение запрещено: разрешите микрофон для Colab в настройках сайта и запустите ячейку снова. Если звук пустой, проверьте выбранный микрофон и сделайте более короткую запись. Текст появляется только после Stop и завершения вычисления (на CPU дольше).

In [ ]:
import base64
import tempfile
from IPython.display import Javascript, display

MAX_RECORD_SECONDS = 30
MAX_AUDIO_BYTES = 5 * 1024 * 1024
last_transcript = ""  # предыдущий результат не переносится в следующую запись

RECORDER_JS = r"""
(async () => {
  // Повторный запуск завершает старую запись и освобождает её микрофон.
  if (window.__gigaamRecorderCancel) window.__gigaamRecorderCancel();
  const maxMs = 30000, maxBytes = 5 * 1024 * 1024;
  const panel = document.createElement('div');
  panel.style.cssText = 'padding:16px;border:1px solid #bbb;border-radius:8px';
  const start = document.createElement('button'); start.textContent = 'Start';
  const stop = document.createElement('button'); stop.textContent = 'Stop';
  const status = document.createElement('p'); status.textContent = 'Готово: нажмите Start';
  panel.append(start, stop, status); document.body.appendChild(panel);
  return await new Promise((resolve, reject) => {
    let stream = null, recorder = null, chunks = [], bytes = 0, done = false;
    let recordTimer = null, watchdog = null;
    function cleanup() {
      clearTimeout(recordTimer); clearTimeout(watchdog);
      if (stream) stream.getTracks().forEach(track => track.stop());
      if (recorder && recorder.state !== 'inactive') {
        try { recorder.stop(); } catch (_) {}
      }
      start.disabled = true; stop.disabled = true;
      window.removeEventListener('pagehide', cancel);
      if (window.__gigaamRecorderCancel === cancel) delete window.__gigaamRecorderCancel;
    }
    function fail(message) {
      if (done) return;
      done = true; cleanup(); chunks = [];
      status.textContent = message; reject(new Error(message));
    }
    function cancel() { fail('Запись отменена. Запустите ячейку снова.'); }
    window.__gigaamRecorderCancel = cancel;
    window.addEventListener('pagehide', cancel, {once: true});
    watchdog = setTimeout(() => fail('Истекло время ожидания. Запустите ячейку снова.'), 120000);
    function stopRecording() {
      if (!recorder || recorder.state === 'inactive') { cancel(); return; }
      stop.disabled = true; status.textContent = 'Завершаем запись…';
      recorder.stop();
      // Освобождаем устройство сразу, не ждём передачи данных в Python.
      if (stream) stream.getTracks().forEach(track => track.stop());
    }
    stop.onclick = stopRecording;
    start.onclick = async () => {
      start.disabled = true;
      try {
        if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia || !window.MediaRecorder)
          throw new Error('Браузер не поддерживает запись. Попробуйте Chrome в Colab.');
        status.textContent = 'Разрешите микрофон в браузере…';
        const acquired = await navigator.mediaDevices.getUserMedia({audio: true, video: false});
        // Разрешение могло прийти уже после отмены или таймаута.
        if (done) { acquired.getTracks().forEach(track => track.stop()); return; }
        stream = acquired;
        const mime = ['audio/webm;codecs=opus', 'audio/webm', 'audio/ogg;codecs=opus', 'audio/mp4']
          .find(type => MediaRecorder.isTypeSupported(type));
        recorder = new MediaRecorder(stream, mime ? {mimeType: mime, audioBitsPerSecond: 64000} : {});
        recorder.ondataavailable = event => {
          if (done || !event.data.size) return;
          bytes += event.data.size;
          if (bytes > maxBytes) { fail('Запись превышает 5 МиБ. Повторите короче.'); return; }
          chunks.push(event.data);
        };
        recorder.onerror = () => fail('Ошибка микрофона. Проверьте разрешения и повторите.');
        recorder.onstop = async () => {
          if (done) return;
          clearTimeout(recordTimer);
          if (stream) stream.getTracks().forEach(track => track.stop());
          try {
            const blob = new Blob(chunks, {type: recorder.mimeType}); chunks = [];
            if (!blob.size || blob.size > maxBytes) throw new Error('Пустая или слишком большая запись.');
            const dataUrl = await new Promise((ok, bad) => {
              const reader = new FileReader(); reader.onload = () => ok(reader.result);
              reader.onerror = () => bad(new Error('Не удалось прочитать запись.'));
              reader.readAsDataURL(blob);
            });
            if (done) return;
            done = true; cleanup(); status.textContent = 'Запись завершена. Распознаём…';
            resolve({base64: dataUrl.split(',')[1], bytes: blob.size});
          } catch (error) { fail(error.message || 'Не удалось завершить запись.'); }
        };
        recorder.start(250);
        status.textContent = 'Идёт запись. Нажмите Stop (автостоп через 30 секунд).';
        recordTimer = setTimeout(stopRecording, maxMs);
        clearTimeout(watchdog);
        watchdog = setTimeout(() => fail('Таймаут завершения записи.'), maxMs + 15000);
      } catch (error) {
        fail(error.name === 'NotAllowedError' ? 'Нет разрешения на микрофон. Разрешите его в настройках сайта.'
          : (error.message || 'Микрофон недоступен.'));
      }
    };
  });
})()
"""

# Только локальные файлы; ограничения декодирования применяет app.asr.normalize_audio.
INFERENCE_CODE = r"""
import json, sys
from pathlib import Path
from app.asr import ASRSettings, LocalASR
settings = ASRSettings(
    model_path=Path(sys.argv[1]), device=sys.argv[2], cpu_threads=2,
    max_audio_seconds=35, max_input_bytes=5 * 1024 * 1024,
    ffmpeg_timeout_seconds=60, temp_root=sys.argv[4],
)
segments = LocalASR(settings).transcribe(Path(sys.argv[3]))
print(json.dumps({"text": " ".join(segment["text"] for segment in segments)}, ensure_ascii=False))
"""
recording = None
raw_audio = None
try:
    recording = output.eval_js(RECORDER_JS)
    encoded = recording.get("base64", "")
    if not isinstance(encoded, str) or len(encoded) > 4 * ((MAX_AUDIO_BYTES + 2) // 3):
        raise ValueError("Недопустимый размер записи")
    raw_audio = base64.b64decode(encoded, validate=True)
    del encoded
    if not 0 < len(raw_audio) <= MAX_AUDIO_BYTES or len(raw_audio) != recording.get("bytes"):
        raise ValueError("Запись пустая или повреждена")
    with tempfile.TemporaryDirectory(prefix="gigaam-mic-") as temporary:
        audio_path = Path(temporary) / "microphone.audio"
        audio_path.write_bytes(raw_audio)
        recording = None
        raw_audio = None
        result = subprocess.run(
            [str(PYTHON), "-c", INFERENCE_CODE, str(MODEL), DEVICE, str(audio_path), temporary],
            cwd=REPO, text=True, capture_output=True, timeout=300,
        )
        if result.returncode:
            raise RuntimeError("Не удалось распознать запись. Проверьте микрофон; "
                               "попробуйте 5–10 секунд или DEVICE = 'cpu' в ячейке 2.")
        last_transcript = json.loads(result.stdout)["text"]
        del result
    print(last_transcript or "Текст не обнаружен. Попробуйте говорить ближе к микрофону.")
except subprocess.TimeoutExpired:
    print("Распознавание превысило 5 минут. Попробуйте короткую запись или GPU.")
finally:
    recording = None
    raw_audio = None

## Фразы для самостоятельной проверки
Запишите каждый пример отдельно, затем попробуйте смешанный. Сравните услышанное и текст вручную, включая пропуски, имена и числа. Эти примеры не измеряют качество модели.

- **RU:** «Здравствуйте. Проверим распознавание русской речи. Следующая встреча состоится завтра в десять утра.»
- **KK:** «Сәлеметсіз бе! Бүгін қазақ тіліндегі сөйлеуді тануды тексереміз. Келесі кездесу ертең сағат онда өтеді.»
- **EN:** “Hello. We are testing English speech recognition. Our next meeting is tomorrow at ten in the morning.”
- **Смешанная речь:** «Сәлем, команда. Сегодня проверяем новый сервис. Please send the report tomorrow. Рақмет!»

Для новой попытки запустите только ячейку 3. Микрофон записывает ваш выбранный аудиовход; notebook не подключается к Teams и не захватывает звук участников встречи автоматически.

## 4. Необязательно: скачать последний текст
Запустите ячейку, затем нажмите **Скачать TXT**. Файл создаётся в браузере; на диск runtime расшифровка не записывается. Скачанный файл останется на вашем устройстве.

In [ ]:
if not last_transcript:
    print("Сначала получите непустую расшифровку в ячейке 3.")
else:
    # JSON-кодирование защищает текст от интерпретации как JavaScript.
    text_json = json.dumps(last_transcript, ensure_ascii=True)
    display(Javascript("""(() => {
      const button = document.createElement('button');
      button.textContent = 'Скачать TXT';
      button.onclick = () => {
        const blob = new Blob([""" + text_json + """], {type: 'text/plain;charset=utf-8'});
        const url = URL.createObjectURL(blob);
        const link = document.createElement('a');
        link.href = url; link.download = 'gigaam-transcript.txt'; link.click();
        setTimeout(() => URL.revokeObjectURL(url), 1000);
      };
      document.body.appendChild(button);
    })();"""))

## 5. Очистить результат
Ячейка удаляет последний текст из переменной и очищает выводы текущей сессии. Для полного сброса памяти выберите **Runtime → Disconnect and delete runtime**. Перед сохранением или публикацией также проверьте **Edit → Clear all outputs**. Удаление вывода не удаляет ранее сохранённые копии, версии или уже скачанный TXT.

**Проверки разработчика:** синтаксис Python и JavaScript, структура notebook и проверки безопасности записи автоматизированы в репозитории. Работу разрешений микрофона и полный запуск именно в Google Colab необходимо подтвердить в вашем браузере; автоматические проверки не заменяют этот тест.

In [ ]:
last_transcript = ""
recording = None
raw_audio = None
if "text_json" in globals():
    del text_json
output.clear()